In [ ]:
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, Trainer, TrainingArguments, AutoModelForSequenceClassification, DataCollatorWithPadding
import evaluate
import torch
import numpy as np

In [ ]:
accuracy = evaluate.load("accuracy")

# hyperparameters from https://github.com/clinc/oos-eval/blob/master/hyperparameters.csv
learning_rate = 4.00e-05
warmup_proportion = 0.1
train_batch_size = 32
eval_batch_size = 32
num_train_epochs = 5
gradient_accumulation_steps = 1

# change o bert-large-uncase
huggingface_modelname = "bert-base-uncased"
huggingface_modelname = "distilbert/distilbert-base-uncased"

TRAIN_SAMPLE_SIZE = int(10e4)
TEST_SAMPLE_SIZE = int(10e3)
# OUTPUT_DIR = "/var/scratch/fht800/hf_home"
OUTPUT_DIR = "test_trainer"

In [ ]:
#TODO: load full dataset, merge train and test splits and create train, calibration, test splits
# dbpedia = load_dataset('fancyzhx/dbpedia_14', split='test')
# dataset = dbpedia.train_test_split(test_size=0.4)
# del dbpedia

dbpedia = load_dataset('fancyzhx/dbpedia_14')
dataset = dbpedia

#TODO FdH: concatenate title tot text input
dataset.remove_columns('title')

In [ ]:
dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(huggingface_modelname)

def tokenize_function(examples):
    return tokenizer(examples["content"], truncation=True)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [ ]:
tokenized_datasets = dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

dataset['train'].features['label'].names, dataset

In [ ]:
labels = tokenized_datasets['train'].features['label'].names
num_labels = len(labels)
id2label = {k: v for k,v in enumerate(labels)}
label2id = {v: k for k,v in id2label.items()}

In [ ]:
label2id

In [ ]:

model = AutoModelForSequenceClassification.from_pretrained(
    huggingface_modelname,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    )
if torch.cuda.is_available(): DEVICE = 'cuda'
elif torch.backends.mps.is_available(): DEVICE = 'mps'
else: DEVICE = 'cpu'
model.to(DEVICE)
torch.mps.empty_cache()
torch.mps.current_allocated_memory(), torch.mps.driver_allocated_memory()

In [ ]:

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=learning_rate,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    gradient_accumulation_steps=gradient_accumulation_steps,
    load_best_model_at_end=True,
    )
metric = evaluate.load("accuracy")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()
trainer.save_model(OUTPUT_DIR)
print("saved model")

In [ ]:
# TODO: evaluate final model on test set
print(model.eval())
print(trainer.evaluate())